# 🔬 Colab GPU: Low-Label Benchmark – VisReg Multi-Scale FPN Fine-Tuned
### Protocol: BraTS 2024 GLI Multiregion Low-Data Benchmark
**Architecture Role**: 3D VisReg JEPA + Multi-Scale FPN Downstream Fine-Tuning (Primary Proposed)

This notebook evaluates label efficiency across **1%, 5%, 10%, 25%, and 50%** annotation budgets using a **modular cell-by-cell execution pipeline**:
1. **Strict Monotonic Nested Sampling**: Every percentage strictly adds up cumulative patient scans ($\mathcal{S}_{1\%} \subset \mathcal{S}_{5\%} \subset \mathcal{S}_{10\%} \subset \mathcal{S}_{25\%} \subset \mathcal{S}_{50\%}$) via deterministic quantile ranking (`seed=42`).
2. **Dedicated Train Cell**: Trains with Early Stopping (`patience=5`, `min_epochs=5`, `max_epochs=50`, `save_freq=10`), saving checkpoints every 10 epochs and the best model (`_best.pt`) to Google Drive.
3. **Dedicated Held-Out Evaluation Cell**: Evaluates on all 242 held-out test volumes, printing WT, TC, ET Dice and HD95 live in the output.
4. **Summary & Curve Plot**: Aggregates all fractions into master Table 2 format and plots the efficiency curve.
5. **1-Click Package**: Archives all fraction outputs to Google Drive.

## 1. Hardware, CUDA & Google Drive Mounting
Verifies GPU accelerator and mounts Google Drive.

In [ ]:
import datetime
import time
from pathlib import Path
import torch

print('=' * 60)
print('GPU Accelerator Status:')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: CUDA not detected! Check Colab Runtime -> Change runtime type -> GPU.')
print('=' * 60)

from google.colab import drive
drive_mounted = Path('/content/drive/MyDrive').exists()
if not drive_mounted:
    drive.mount('/content/drive', force_remount=False)
print(f'Google Drive mounted: {Path("/content/drive/MyDrive").exists()}')

## 2. Standardized Configuration & Early Stopping Parameters
Defines deterministic SEED=42 (guaranteeing nested cumulative subsets), patience=5, and Drive destinations.

In [ ]:
# =========================================================================
# ⚙️ Standardized Low-Label Experiment Configuration
# =========================================================================
SEED = 42                 # Deterministic seed guaranteeing monotonic nested sampling
NUM_WORKERS = 4          # Parallel DataLoader workers
BATCH_SIZE = 4          # Volumetric downstream batch size
PATCH_SIZE = 8           # 8³ voxel patch (4,096 tokens)
MAX_EPOCHS = 50          # Maximum training epochs ceiling
PATIENCE = 5             # Early stopping patience (epochs without validation improvement)
MIN_EPOCHS = 5           # Burn-in epochs before early stopping can trigger
MIN_STEPS = 250          # Step floor to prevent undertraining on tiny fractions
SAVE_FREQ = 10           # Periodic checkpoint cadence (saves _epoch_10.pt, _epoch_20.pt, etc.)

TARGET_DRIVE_FOLDER = 'thesis/low_label/visreg_multiscale'
ZIP_NAME = 'visreg_multiscale_low_data.zip'
MODEL_ALIAS = 'visreg_multiscale'

output_dir = Path(f'/content/drive/MyDrive/{TARGET_DRIVE_FOLDER}')
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Model Alias:         {MODEL_ALIAS}')
print(f'Target Drive Folder: {output_dir}')
print(f'Early Stopping:      patience={PATIENCE}, min_epochs={MIN_EPOCHS}, max_epochs={MAX_EPOCHS}')
print(f'Checkpoint Cadence:  save_freq={SAVE_FREQ}, seed={SEED}')

## 3. Dependencies Installation
Installs core medical imaging dependencies.

In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate matplotlib pandas
import monai
import nibabel as nib
print(f'MONAI: {monai.__version__} | NiBabel: {nib.__version__}')

## 4. Codebase Setup & Editable Installation
Discovers repository and performs editable install.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

code_candidates = [
    Path('/content/thesis_3d'),
    Path('/content/drive/MyDrive/thesis/thesis_3d'),
    Path.cwd(),
]
repo_root = None
for cand in code_candidates:
    if (cand / 'src' / 'brats_jepa_3d').exists():
        repo_root = cand
        break

if repo_root is None or not (repo_root / 'src' / 'brats_jepa_3d').exists():
    drive_repo = Path('/content/drive/MyDrive/thesis/thesis_3d')
    if drive_repo.exists():
        repo_root = Path('/content/thesis_3d')
        if not repo_root.exists():
            shutil.copytree(drive_repo, repo_root)
    else:
        repo_root = Path('/content/thesis_3d')

os.chdir(str(repo_root))
print(f'Working Directory: {Path.cwd()}')
!pip install -q -e .

## 5. Dataset Ingestion & Discovery
Resolves preprocessed BraTS 3D volumes on fast local NVMe SSD or Google Drive.

In [ ]:
import os
from pathlib import Path

data_candidates = [
    Path('/content/data/brats_gli_3d_v2'),
    Path('/content/data'),
    Path('/content/drive/MyDrive/thesis/data/brats_gli_3d_v2'),
    Path('/content/drive/MyDrive/thesis/data'),
    Path('data/processed/brats_gli_3d_v2'),
]
data_dir = None
for cand in data_candidates:
    if cand.exists() and (cand / 'metadata.csv').exists():
        data_dir = cand
        break

if data_dir is None:
    drive_zip = Path('/content/drive/MyDrive/thesis/data/brats_gli_3d_v2.zip')
    if drive_zip.exists():
        print(f'Unpacking {drive_zip} to /content/data/ ...')
        !unzip -q -n "{drive_zip}" -d /content/data/
        data_dir = Path('/content/data/brats_gli_3d_v2')

print(f'Dataset directory resolved: {data_dir}')
assert data_dir is not None and data_dir.exists(), 'Dataset not found!'

## 6. Pre-trained Checkpoint Intake
Locates the pre-trained 3D VisReg JEPA encoder checkpoint from Google Drive.

In [ ]:
from pathlib import Path
from brats_jepa_3d.config import CHECKPOINTS_DIR
from brats_jepa_3d.utils import sort_checkpoints_by_epoch

pretrain_dir = Path('/content/drive/MyDrive/thesis/pretrain model')
candidates = []

if pretrain_dir.exists():
    for folder in [pretrain_dir / 'checkpoints', pretrain_dir]:
        if folder.is_dir():
            best = folder / 'visreg_jepa_3d_best.pt'
            if best.is_file():
                candidates.append(best)
            epochs = sort_checkpoints_by_epoch([p for p in folder.glob('visreg_jepa*_epoch_*.pt') if p.is_file()])
            if epochs:
                candidates.append(epochs[-1])
            general = [p for p in sorted(folder.glob('visreg_jepa*.pt')) if p.is_file() and 'scratch' not in p.name and 'frozen' not in p.name]
            candidates.extend(general)

if not candidates:
    for folder in [CHECKPOINTS_DIR, Path('outputs/checkpoints')]:
        if folder.is_dir():
            best = folder / 'visreg_jepa_3d_best.pt'
            if best.is_file():
                candidates.append(best)

pretrained_ckpt = str(candidates[0]) if candidates else None
assert pretrained_ckpt, (
    f'❌ Pre-trained checkpoint not found in /content/drive/MyDrive/thesis/pretrain model/!\n'
    f'Please verify your pre-trained .pt file exists in Google Drive under thesis/pretrain model/.'
)
print(f'✓ Found pre-trained encoder checkpoint: {pretrained_ckpt}')

## 7. Annotation Budget: 1% (12 Patient Volumes)
**Monotonic Subset**: 1% Annotation Budget (12 Patient Volumes)

### 7.1 Train Cell (1%)
Trains with Early Stopping (`patience=5`, `min_epochs=5`, `max_epochs=50`, `save_freq=10`).
Periodic checkpoints are saved every 10 epochs, and the best model is saved to `visreg_multiscale_0_01_best.pt`.

In [ ]:
# Train Cell (1% labels, 12 volumes)
frac_dir = output_dir / 'frac_0_01'
frac_dir.mkdir(parents=True, exist_ok=True)
print(f'Training 1% in: {frac_dir}')

!python scripts/train_downstream_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{frac_dir}" \
    --fraction 0.01 \
    --seed {SEED} \
    --epochs {MAX_EPOCHS} \
    --patience {PATIENCE} \
    --min_epochs {MIN_EPOCHS} \
    --min_steps {MIN_STEPS} \
    --save_freq {SAVE_FREQ} \
    --ckpt_prefix "{MODEL_ALIAS}_0_01" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --model_type visreg_jepa --decoder_type multiscale --pretrained_checkpoint "{pretrained_ckpt}" --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} --deep_supervision \
    --amp

### 7.2 Held-Out Test Evaluation Cell (1%)
Evaluates the saved `visreg_multiscale_0_01_best.pt` across all **242 held-out test volumes**.

In [ ]:
# Eval Cell (1% best checkpoint -> held-out test split)
frac_dir = output_dir / 'frac_0_01'
eval_dir = frac_dir / 'eval'
ckpt_path = frac_dir / 'checkpoints' / f'{MODEL_ALIAS}_0_01_best.pt'
assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'

print(f'Evaluating checkpoint: {ckpt_path}')
!python scripts/evaluate_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{eval_dir}" \
    --checkpoint "{ckpt_path}" \
    --model_type visreg_jepa --decoder_type multiscale --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --batch_size 1 \
    --num_workers {NUM_WORKERS} \
    --amp

## 8. Annotation Budget: 5% (58 Patient Volumes)
**Monotonic Subset**: 5% Annotation Budget (58 Patient Volumes = 12 from 1% + 46 new)

### 8.1 Train Cell (5%)
Trains with Early Stopping (`patience=5`, `min_epochs=5`, `max_epochs=50`, `save_freq=10`).
Periodic checkpoints are saved every 10 epochs, and the best model is saved to `visreg_multiscale_0_05_best.pt`.

In [ ]:
# Train Cell (5% labels, 58 volumes)
frac_dir = output_dir / 'frac_0_05'
frac_dir.mkdir(parents=True, exist_ok=True)
print(f'Training 5% in: {frac_dir}')

!python scripts/train_downstream_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{frac_dir}" \
    --fraction 0.05 \
    --seed {SEED} \
    --epochs {MAX_EPOCHS} \
    --patience {PATIENCE} \
    --min_epochs {MIN_EPOCHS} \
    --min_steps {MIN_STEPS} \
    --save_freq {SAVE_FREQ} \
    --ckpt_prefix "{MODEL_ALIAS}_0_05" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --model_type visreg_jepa --decoder_type multiscale --pretrained_checkpoint "{pretrained_ckpt}" --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} --deep_supervision \
    --amp

### 8.2 Held-Out Test Evaluation Cell (5%)
Evaluates the saved `visreg_multiscale_0_05_best.pt` across all **242 held-out test volumes**.

In [ ]:
# Eval Cell (5% best checkpoint -> held-out test split)
frac_dir = output_dir / 'frac_0_05'
eval_dir = frac_dir / 'eval'
ckpt_path = frac_dir / 'checkpoints' / f'{MODEL_ALIAS}_0_05_best.pt'
assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'

print(f'Evaluating checkpoint: {ckpt_path}')
!python scripts/evaluate_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{eval_dir}" \
    --checkpoint "{ckpt_path}" \
    --model_type visreg_jepa --decoder_type multiscale --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --batch_size 1 \
    --num_workers {NUM_WORKERS} \
    --amp

## 9. Annotation Budget: 10% (115 Patient Volumes)
**Monotonic Subset**: 10% Annotation Budget (115 Patient Volumes = 58 from 5% + 57 new)

### 9.1 Train Cell (10%)
Trains with Early Stopping (`patience=5`, `min_epochs=5`, `max_epochs=50`, `save_freq=10`).
Periodic checkpoints are saved every 10 epochs, and the best model is saved to `visreg_multiscale_0_10_best.pt`.

In [ ]:
# Train Cell (10% labels, 115 volumes)
frac_dir = output_dir / 'frac_0_10'
frac_dir.mkdir(parents=True, exist_ok=True)
print(f'Training 10% in: {frac_dir}')

!python scripts/train_downstream_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{frac_dir}" \
    --fraction 0.1 \
    --seed {SEED} \
    --epochs {MAX_EPOCHS} \
    --patience {PATIENCE} \
    --min_epochs {MIN_EPOCHS} \
    --min_steps {MIN_STEPS} \
    --save_freq {SAVE_FREQ} \
    --ckpt_prefix "{MODEL_ALIAS}_0_10" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --model_type visreg_jepa --decoder_type multiscale --pretrained_checkpoint "{pretrained_ckpt}" --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} --deep_supervision \
    --amp

### 9.2 Held-Out Test Evaluation Cell (10%)
Evaluates the saved `visreg_multiscale_0_10_best.pt` across all **242 held-out test volumes**.

In [ ]:
# Eval Cell (10% best checkpoint -> held-out test split)
frac_dir = output_dir / 'frac_0_10'
eval_dir = frac_dir / 'eval'
ckpt_path = frac_dir / 'checkpoints' / f'{MODEL_ALIAS}_0_10_best.pt'
assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'

print(f'Evaluating checkpoint: {ckpt_path}')
!python scripts/evaluate_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{eval_dir}" \
    --checkpoint "{ckpt_path}" \
    --model_type visreg_jepa --decoder_type multiscale --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --batch_size 1 \
    --num_workers {NUM_WORKERS} \
    --amp

## 10. Annotation Budget: 25% (286 Patient Volumes)
**Monotonic Subset**: 25% Annotation Budget (286 Patient Volumes = 115 from 10% + 171 new)

### 10.1 Train Cell (25%)
Trains with Early Stopping (`patience=5`, `min_epochs=5`, `max_epochs=50`, `save_freq=10`).
Periodic checkpoints are saved every 10 epochs, and the best model is saved to `visreg_multiscale_0_25_best.pt`.

In [ ]:
# Train Cell (25% labels, 286 volumes)
frac_dir = output_dir / 'frac_0_25'
frac_dir.mkdir(parents=True, exist_ok=True)
print(f'Training 25% in: {frac_dir}')

!python scripts/train_downstream_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{frac_dir}" \
    --fraction 0.25 \
    --seed {SEED} \
    --epochs {MAX_EPOCHS} \
    --patience {PATIENCE} \
    --min_epochs {MIN_EPOCHS} \
    --min_steps {MIN_STEPS} \
    --save_freq {SAVE_FREQ} \
    --ckpt_prefix "{MODEL_ALIAS}_0_25" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --model_type visreg_jepa --decoder_type multiscale --pretrained_checkpoint "{pretrained_ckpt}" --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} --deep_supervision \
    --amp

### 10.2 Held-Out Test Evaluation Cell (25%)
Evaluates the saved `visreg_multiscale_0_25_best.pt` across all **242 held-out test volumes**.

In [ ]:
# Eval Cell (25% best checkpoint -> held-out test split)
frac_dir = output_dir / 'frac_0_25'
eval_dir = frac_dir / 'eval'
ckpt_path = frac_dir / 'checkpoints' / f'{MODEL_ALIAS}_0_25_best.pt'
assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'

print(f'Evaluating checkpoint: {ckpt_path}')
!python scripts/evaluate_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{eval_dir}" \
    --checkpoint "{ckpt_path}" \
    --model_type visreg_jepa --decoder_type multiscale --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --batch_size 1 \
    --num_workers {NUM_WORKERS} \
    --amp

## 11. Annotation Budget: 50% (572 Patient Volumes)
**Monotonic Subset**: 50% Annotation Budget (572 Patient Volumes = 286 from 25% + 286 new)

### 11.1 Train Cell (50%)
Trains with Early Stopping (`patience=5`, `min_epochs=5`, `max_epochs=50`, `save_freq=10`).
Periodic checkpoints are saved every 10 epochs, and the best model is saved to `visreg_multiscale_0_50_best.pt`.

In [ ]:
# Train Cell (50% labels, 572 volumes)
frac_dir = output_dir / 'frac_0_50'
frac_dir.mkdir(parents=True, exist_ok=True)
print(f'Training 50% in: {frac_dir}')

!python scripts/train_downstream_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{frac_dir}" \
    --fraction 0.5 \
    --seed {SEED} \
    --epochs {MAX_EPOCHS} \
    --patience {PATIENCE} \
    --min_epochs {MIN_EPOCHS} \
    --min_steps {MIN_STEPS} \
    --save_freq {SAVE_FREQ} \
    --ckpt_prefix "{MODEL_ALIAS}_0_50" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --model_type visreg_jepa --decoder_type multiscale --pretrained_checkpoint "{pretrained_ckpt}" --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} --deep_supervision \
    --amp

### 11.2 Held-Out Test Evaluation Cell (50%)
Evaluates the saved `visreg_multiscale_0_50_best.pt` across all **242 held-out test volumes**.

In [ ]:
# Eval Cell (50% best checkpoint -> held-out test split)
frac_dir = output_dir / 'frac_0_50'
eval_dir = frac_dir / 'eval'
ckpt_path = frac_dir / 'checkpoints' / f'{MODEL_ALIAS}_0_50_best.pt'
assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'

print(f'Evaluating checkpoint: {ckpt_path}')
!python scripts/evaluate_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{eval_dir}" \
    --checkpoint "{ckpt_path}" \
    --model_type visreg_jepa --decoder_type multiscale --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --batch_size 1 \
    --num_workers {NUM_WORKERS} \
    --amp

## 12. Master Table 2 Summary & Efficiency Curve
Compiles held-out test metrics across all fractions into the canonical Table 2 format, saves `low_data_3d_summary.csv` and `.md`, and renders the efficiency curve.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

fractions_info = [
    ('0_01', '1%', 12),
    ('0_05', '5%', 58),
    ('0_10', '10%', 115),
    ('0_25', '25%', 286),
    ('0_50', '50%', 572),
]

summary_rows = []
for frac_suffix, pct_label, n_vols in fractions_info:
    eval_csv = output_dir / f'frac_{frac_suffix}' / 'eval' / 'metrics' / 'benchmark_3d_summary.csv'
    if eval_csv.exists():
        df_eval = pd.read_csv(eval_csv)
        row_dict = {
            'Fraction': pct_label,
            'Volumes': n_vols,
            'WT Dice (%)': 'n/a',
            'TC Dice (%)': 'n/a',
            'ET Dice (%)': 'n/a',
            'Mean Dice (%)': 'n/a',
            'Mean HD95 (voxels)': 'n/a',
        }
        dice_vals = []
        hd95_vals = []
        for rname in ['WT', 'TC', 'ET']:
            sub = df_eval[df_eval['Model'].str.contains(f'\\[{rname}\\]', na=False)]
            if not sub.empty:
                d_str = str(sub.iloc[0]['Dice (%)'])
                row_dict[f'{rname} Dice (%)'] = d_str
                try:
                    val = float(d_str.split('±')[0].strip())
                    dice_vals.append(val)
                except Exception:
                    pass
                if 'HD95 (voxels)' in sub.columns:
                    h_str = str(sub.iloc[0]['HD95 (voxels)'])
                    try:
                        h_val = float(h_str.split('±')[0].strip())
                        hd95_vals.append(h_val)
                    except Exception:
                        pass
        if dice_vals:
            row_dict['Mean Dice (%)'] = f'{sum(dice_vals) / len(dice_vals):.2f}'
        if hd95_vals:
            row_dict['Mean HD95 (voxels)'] = f'{sum(hd95_vals) / len(hd95_vals):.2f}'
        summary_rows.append(row_dict)

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    metrics_dir = output_dir / 'metrics'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    summary_csv = metrics_dir / 'low_data_3d_summary.csv'
    summary_md = metrics_dir / 'low_data_3d_summary.md'
    summary_df.to_csv(summary_csv, index=False)
    try:
        summary_df.to_markdown(summary_md, index=False)
    except Exception:
        with open(summary_md, 'w') as f:
            f.write(summary_df.to_string(index=False))

    print('\n' + '=' * 70)
    print(f'📊 LOW-DATA LABEL EFFICIENCY SUMMARY: {MODEL_ALIAS}')
    print('=' * 70)
    print(summary_df.to_string(index=False))
    print('=' * 70)
    print(f'✓ Saved summary CSV to: {summary_csv}')
    print(f'✓ Saved summary Markdown to: {summary_md}')

    valid_pts = []
    for _, r in summary_df.iterrows():
        try:
            m_val = float(r['Mean Dice (%)'])
            valid_pts.append((r['Fraction'], r['Volumes'], m_val))
        except (ValueError, TypeError):
            pass
    if valid_pts:
        labels, vols, y_vals = zip(*valid_pts)
        plt.figure(figsize=(8, 5))
        plt.plot(labels, y_vals, marker='o', linewidth=2.5, markersize=8, color='#1f77b4', label=MODEL_ALIAS)
        for lbl, y in zip(labels, y_vals):
            plt.annotate(f'{y:.2f}%', (lbl, y), textcoords='offset points', xytext=(0, 10), ha='center', fontweight='bold')
        plt.title(f'Label Efficiency: {MODEL_ALIAS} (BraTS 2024 Held-Out Test)', fontsize=13, fontweight='bold')
        plt.xlabel('Annotation Fraction (Patient Volume Budget)', fontsize=11)
        plt.ylabel('Macro Test Dice (%) [WT/TC/ET]', fontsize=11)
        plt.ylim(0, 100)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(loc='lower right')
        plt.tight_layout()
        plot_path = metrics_dir / 'low_data_curve.png'
        plt.savefig(plot_path, dpi=300)
        print(f'✓ Saved efficiency curve plot to: {plot_path}')
        plt.show()
else:
    print('⚠️ No evaluation results found yet. Execute the Train and Eval cells above!')

## 13. 1-Click Package & Export to Google Drive
Archives all fraction checkpoints, logs, and evaluation reports into a standalone zip file in Google Drive.

In [ ]:
import shutil
from pathlib import Path

zip_path = Path('/content/drive/MyDrive/thesis/low_label') / ZIP_NAME
zip_path.parent.mkdir(parents=True, exist_ok=True)

print(f'Creating archive {zip_path} from {output_dir} ...')
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', output_dir)
print(f'✓ Export completed: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')